## Observer

---

> **In one line.** A subject $S$ keeps a set of observers $\mathcal{O} = \{O_1, \ldots, O_n\}$ and, whenever its state changes, runs a single broadcast $\mathrm{notify}$ that pushes the new state to *every* $O_i \in \mathcal{O}$ — a one-to-many dependency that keeps subscribers in sync without $S$ knowing who they are.

### 1. The subject and its observers

At the centre sits one distinguished object, the **subject** (or *observable*) $S$. It carries a piece of mutable data we call its **state**, written $\mathrm{state}(S)$ — this is the value that changes and that everyone else cares about. Around $S$ orbits a **set of registered observers**

$$\mathcal{O} = \{O_1, O_2, \ldots, O_n\},$$

where each **observer** $O_i$ is a single subscriber that implements one agreed-upon method, $\mathrm{update}(\cdot)$. Membership in this set *is* the subscription: writing $O_i \in \mathcal{O}$ means "$O_i$ is currently listening to $S$," and removing $O_i$ from $\mathcal{O}$ is exactly unsubscribing. The set is not fixed at construction time; it grows and shrinks while the program runs.

### 2. The broadcast

The operation that defines the pattern is $\mathrm{notify}$: a **broadcast function** that $S$ invokes on itself the moment $\mathrm{state}(S)$ changes. A single call fans out to the whole set —

$$S \;\xrightarrow{\;\mathrm{notify}\;}\; \{O_1, O_2, \ldots, O_n\},$$

and what it does at each recipient is to hand them the current state through their `update()` method. The defining law is the *universally quantified* dispatch: no observer is special, none is skipped.

$$\boxed{\;\forall\, O_i \in \mathcal{O},\quad O_i.\mathrm{update}\big(\mathrm{state}(S)\big)\;}$$

The $\forall i$ is the heart of it — one mutation of $S$ produces $n$ synchronized updates. As a data-flow chain, a change at the source propagates outward to every subscriber at once:

$$\mathrm{state}(S)\ \text{changes} \;\xrightarrow{\;\mathrm{notify}\;}\; \forall i \;\xrightarrow{\;\mathrm{update}\;}\; O_i\big(\mathrm{state}(S)\big).$$

### 3. Direction and decoupling

The dependency points **one way**: $S$ pushes to $\mathcal{O}$, never the reverse. Crucially $S$ knows each $O_i$ only as "something that has an `update()` method" — it has no idea what any observer *does* with the state. This is the source of the pattern's flexibility, and it rests on three conditions.

1. **Decoupling.** $S$ depends only on the `update()` contract, not on any concrete observer. Each $O_i$ is interchangeable as long as it honours $O_i.\mathrm{update} : \mathrm{state}(S) \to (\text{anything})$; what happens after the call is invisible to $S$.
2. **Open–closed.** New observers are added purely by insertion into the set, $\mathcal{O} \mapsto \mathcal{O} \cup \{O_{n+1}\}$, with **no change to $S$**. The subject is closed for modification yet open for new subscribers.
3. **Dynamic membership.** $\mathcal{O}$ is mutable at runtime: observers may subscribe ($\mathcal{O} \cup \{O\}$) and unsubscribe ($\mathcal{O} \setminus \{O\}$) at any moment, and the next $\mathrm{notify}$ simply ranges over whatever set is current.

> 📰 A newspaper ($S$) publishes a new edition. Every subscriber ($O_i \in \mathcal{O}$) gets a copy automatically. The newspaper has no idea what each subscriber does with it — it just delivers to all of them.

### Exercise 01 — Stock Price Alert

---

**Scenario:** A `Stock` object ($S$) tracks a price. Multiple observers ($\mathcal{O}$) — `EmailAlert`, `SMSAlert`, `Dashboard` — need notifying on every price change.

**Your task:** Implement `Stock` and three observer classes. When `stock.set_price()` is called, all $O_i \in \mathcal{O}$ receive `update(price)` automatically.

```python
stock = Stock("AAPL")
stock.subscribe(EmailAlert())   # O_1 ∈ O
stock.subscribe(SMSAlert())     # O_2 ∈ O
stock.set_price(182.5)          # notify → ∀i, O_i.update(state(S))
```

**Hints**

- $S$ holds `self._observers = []` — this is $\mathcal{O}$. `subscribe(o)` appends; `unsubscribe(o)` removes. `_notify()` loops and calls `o.update(state)`.
- Test the open-closed condition: add a new `LoggerObserver` without changing any existing code — just subscribe it.

In [ ]:
# --------------------------------
# Observers (each O_i implements update(state)) — you do not change the subject for these

class EmailAlert:
    def update(self, price):                     # O_i.update(state(S))
        print(f"EmailAlert: price is now {price}")

class SMSAlert:
    def update(self, price):                     # O_i.update(state(S))
        print(f"SMSAlert: price is now {price}")

class Dashboard:
    def update(self, price):                     # O_i.update(state(S))
        print(f"Dashboard: refreshing chart with {price}")

# --------------------------------
# Subject S — your task: hold O, subscribe/unsubscribe, and notify on state change

class Stock:
    def __init__(self, symbol):
        self._symbol = symbol
        self._price = None                       # state(S)
        self._observers = []                     # O = {O_1, ..., O_n}

    def subscribe(self, observer):               # insert O_i into O
        ...

    def unsubscribe(self, observer):             # remove O_i from O
        ...

    def _notify(self):                           # notify: ∀i, O_i.update(state(S))
        ...

    def set_price(self, price):                  # change state(S), then notify
        # self._price = price  (this is state(S))
        # then broadcast to every O_i ∈ O
        ...

# --------------------------------
stock = Stock("AAPL")
stock.subscribe(EmailAlert())    # O_1 ∈ O
stock.subscribe(SMSAlert())      # O_2 ∈ O
stock.subscribe(Dashboard())     # O_3 ∈ O
stock.set_price(182.5)           # notify → ∀i, O_i.update(state(S))

### Exercise 02 — Named Event System

---

**Scenario:** Build a generic `EventEmitter` where observers subscribe to *named events*. Instead of one $\mathcal{O}$, there is $\mathcal{O}_e$ per event $e$ — only subscribers of event $e$ receive it.

**Your task:** Implement `EventEmitter` with `on(event, handler)` and `emit(event, data)`.

```python
emitter = EventEmitter()
emitter.on("login", lambda u: print(f"Welcome {u}"))
emitter.emit("login", "Sadiah")   # only "login" subscribers notified
```

**Hints**

- Store `self._handlers = defaultdict(list)`. Each key is an event name; each value is $\mathcal{O}_e$ — the list of handlers for that event.

In [ ]:
from collections import defaultdict

# --------------------------------
# Subject S with per-event observer sets — your task: implement on() and emit()

class EventEmitter:
    def __init__(self):
        self._handlers = defaultdict(list)       # event e -> O_e (its handlers)

    def on(self, event, handler):                # insert handler into O_e
        # append handler to the list for this event name
        ...

    def emit(self, event, data):                 # notify only O_e: ∀ O_i ∈ O_e, O_i.update(data)
        # loop over handlers registered for `event` and call each with `data`
        ...

# --------------------------------
emitter = EventEmitter()
emitter.on("login", lambda u: print(f"Welcome {u}"))
emitter.on("logout", lambda u: print(f"Goodbye {u}"))
emitter.emit("login", "Sadiah")    # only "login" subscribers (O_login) notified